# U2T02: Sentence Embedding Model Evaluation & Benchmark Analysis
**Author:** Rivaldo (Evaluation, Alignment & Uniformity Lead)  
**Project:** SimCSE Contrastive Learning of Sentence Embeddings  
---
## Overview
This notebook provides a comprehensive evaluation harness for sentence embedding models on the **STS-Benchmark (STS-B)** dataset. It implements:
1. **Baseline Sanity Checks:** Verification against reference Spearman correlation targets ($59.31/47.29$ for Raw BERT, $80.77/76.98$ for SBERT-2019).
2. **Alignment & Uniformity Metrics:** Quantitative hypersphere quality metrics from Wang & Isola (ICML 2020).
3. **Cosine Similarity Density Distributions:** Seaborn KDE plots grouped by STS-B human score brackets $[0-1], [1-2], [2-3], [3-4], [4-5]$.
4. **Nearest-Neighbor Retrieval & Failure Case Analysis:** Identification of lexical overlap vs. semantic contradiction failure cases.

In [1]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure repository root is in python path
sys.path.insert(0, os.path.abspath("."))

from src.data import load_stsb_dataset
from src.metrics import compute_cosine_similarity, compute_spearman_correlation, compute_alignment, compute_uniformity, evaluate_sts_benchmark
from src.evaluate import run_evaluation
from src.analysis import generate_similarity_distribution_plot, analyze_nearest_neighbors_and_failures

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU Device: NVIDIA GeForce RTX 4050 Laptop GPU


## 1. Baseline Sanity Checks (Raw BERT vs. SBERT-2019)
Verifying reference values on STS-B Dev (1,500 pairs) and Test (1,379 pairs) splits before training.

In [2]:
print(">>> Evaluating Raw bert-base-uncased Baseline...")
raw_bert_results = run_evaluation(model_path_or_name="bert-base-uncased", model_type="raw_bert", pooling="mean")

print("\n>>> Evaluating SBERT-2019 Baseline...")
sbert_results = run_evaluation(model_path_or_name="sentence-transformers/bert-base-nli-mean-tokens", model_type="sbert", pooling="mean")

>>> Evaluating Raw bert-base-uncased Baseline...

--- Loading STS-B Dataset ---


Loaded 1500 dev pairs and 1379 test pairs.
Evaluating Model: bert-base-uncased (Type: raw_bert, Pooling: mean, Device: cuda)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



EVALUATION RESULTS FOR bert-base-uncased
Dev  Spearman (x100): 59.31 | Alignment: 0.3678 | Uniformity: -1.6348
Test Spearman (x100): 47.29 | Alignment: 0.3044 | Uniformity: -1.6186


>>> Evaluating SBERT-2019 Baseline...

--- Loading STS-B Dataset ---


Loaded 1500 dev pairs and 1379 test pairs.
Evaluating Model: sentence-transformers/bert-base-nli-mean-tokens (Type: sbert, Pooling: mean, Device: cuda)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


EVALUATION RESULTS FOR sentence-transformers/bert-base-nli-mean-tokens
Dev  Spearman (x100): 80.77 | Alignment: 0.6996 | Uniformity: -3.0557
Test Spearman (x100): 76.98 | Alignment: 0.5498 | Uniformity: -3.0493



## 2. Benchmark Summary Table
Comparing empirical Spearman correlation, Alignment ($\alpha=2$), and Uniformity ($t=2$) against assignment target reference values.

In [3]:
summary_data = [
    {
        "Model": "Raw bert-base-uncased (mean pooling)",
        "Dev Spearman (Target)": 59.31,
        "Dev Spearman (Actual)": raw_bert_results["dev"]["spearman"],
        "Test Spearman (Target)": 47.29,
        "Test Spearman (Actual)": raw_bert_results["test"]["spearman"],
        "Dev Alignment": raw_bert_results["dev"]["alignment"],
        "Dev Uniformity": raw_bert_results["dev"]["uniformity"],
    },
    {
        "Model": "SBERT-2019 (bert-base-nli-mean-tokens)",
        "Dev Spearman (Target)": 80.77,
        "Dev Spearman (Actual)": sbert_results["dev"]["spearman"],
        "Test Spearman (Target)": 76.98,
        "Test Spearman (Actual)": sbert_results["test"]["spearman"],
        "Dev Alignment": sbert_results["dev"]["alignment"],
        "Dev Uniformity": sbert_results["dev"]["uniformity"],
    }
]
df_summary = pd.DataFrame(summary_data)
df_summary

,Model,Dev Spearman (Target),Dev Spearman (Actual),Test Spearman (Target),Test Spearman (Actual),Dev Alignment,Dev Uniformity
0,Raw bert-base-uncased (mean pooling),59.31,59.31,47.29,47.29,0.3678,-1.6348
1,SBERT-2019 (bert-base-nli-mean-tokens),80.77,80.77,76.98,76.98,0.6996,-3.0557


## 3. Cosine Similarity Density Distributions
Partitioning cosine similarity density plots by STS-B human score intervals $[0-1], [1-2], [2-3], [3-4], [4-5]$ to visualize representation collapse vs. semantic separation.

In [4]:
generate_similarity_distribution_plot(
    model_path_or_name="bert-base-uncased",
    output_path="reports/figures/similarity_distribution_bert.png",
    device="cuda" if torch.cuda.is_available() else "cpu",
    pooling="mean"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved similarity distribution plot to: reports/figures/similarity_distribution_bert.png


## 4. Failure Case & Retrieval Analysis
Identifying sentence pairs where raw BERT fails due to high lexical overlap with contradictory semantics.

In [5]:
retrieval_res = analyze_nearest_neighbors_and_failures(
    model_path_or_name="bert-base-uncased",
    output_path="reports/figures/retrieval_analysis_bert.json",
    device="cuda" if torch.cuda.is_available() else "cpu",
    pooling="mean"
)

print("=== TOP RETRIEVAL FAILURE CASES (HIGH LEXICAL OVERLAP, LOW HUMAN SIMILARITY) ===\n")
for idx, item in enumerate(retrieval_res["sample_failures"], 1):
    print(f"{idx}. Sentence 1: {item['sentence1']}")
    print(f"   Sentence 2: {item['sentence2']}")
    print(f"   Human Score: {item['human_score']} / 5.0")
    print(f"   Raw BERT Cosine Similarity: {item['cosine_similarity']:.4f}")
    print(f"   Reason: {item['discrepancy_explanation']}\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved retrieval analysis to: reports/figures/retrieval_analysis_bert.json
=== TOP RETRIEVAL FAILURE CASES (HIGH LEXICAL OVERLAP, LOW HUMAN SIMILARITY) ===

1. Sentence 1: A woman is cutting an onion.
   Sentence 2: A woman is cleaning a garden.
   Human Score: 0.05 / 5.0
   Raw BERT Cosine Similarity: 0.9327
   Reason: High lexical overlap with contradictory semantics

2. Sentence 1: A woman is dancing.
   Sentence 2: A woman is playing violin.
   Human Score: 0.08 / 5.0
   Raw BERT Cosine Similarity: 0.9426
   Reason: High lexical overlap with contradictory semantics

3. Sentence 1: Gunmen kill nine people in northwest Pakistan
   Sentence 2: Gunmen kill 3 policemen in Iraq
   Human Score: 0.08 / 5.0
   Raw BERT Cosine Similarity: 0.9169
   Reason: High lexical overlap with contradictory semantics

4. Sentence 1: A man is riding a motorcycle.
   Sentence 2: A woman is riding a horse.
   Human Score: 0.15 / 5.0
   Raw BERT Cosine Similarity: 0.9447
   Reason: High lexical overlap with 